# HW3 — Two retrievers and a model: what RAG actually sends

**Name:**
**Student ID:**
**Group:**
**Repository:**

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

>

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. | 9 |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. | 10 |
| D3 | Embeddings represent text as numerical vectors. | 6 |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl = (9 + 10 + 6) / 3 = 25/3 ≈ 8,333**

**Query terms: how, does, rag, retrieve, information**

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3,5/0,5) = 1,9459 |
| does | 0 | 0 | 0 | 0 | 1,9459 |
| rag | 0 | 0 | 0 | 0 | 1,9459 |
| retrieve | 0 | 0 | 0 | 0 | 1,9459 |
| information | 0 | 0 | 0 | 0 | 1,9459 |

**BM25 of each document**: Normalised length factor: `k1 · (1 − b + b · |d| / avgdl)` with k1 = 1.5, b = 0.75, avgdl = 25/3 ≈ 8.333

- D1: 1,5 · (0,25 + 0,75 · 9/8,333) = 1,5 · 1,06 = 1,59
- D2: 1,5 · (0,25 + 0,75 · 10/8,333) = 1,5 · 1,15 = 1,725
- D3: 1,5 · (0,25 + 0,75 · 6/8,333) = 1,5 · 0,79 = 1,185

Every query term has f(t, d) = 0 in every document, so the numerator `idf(t) · f(t,d) · (k1 + 1)` is 0 for each term. No term contributes to any score.

**D1**
- how: 1,9459 · 0 · 2,5 / (0 + 1,59) = 0
- does: 1,9459 · 0 · 2,5 / (0 + 1,59) = 0
- rag: 1,9459 · 0 · 2,5 / (0 + 1,59) = 0
- retrieve: 1,9459 · 0 · 2,5 / (0 + 1,59) = 0
- information: 1,9459 · 0 · 2,5 / (0 + 1,59) = 0
- score(q, D1) = 0 + 0 + 0 + 0 + 0 = **0**

**D2**
- how: 1,9459 · 0 · 2,5 / (0 + 1,725) = 0
- does: 1,9459 · 0 · 2,5 / (0 + 1,725) = 0
- rag: 1,9459 · 0 · 2,5 / (0 + 1,725) = 0
- retrieve: 1,9459 · 0 · 2,5 / (0 + 1,725) = 0
- information: 1,9459 · 0 · 2,5 / (0 + 1,725) = 0
- score(q, D2) = 0 + 0 + 0 + 0 + 0 = **0**

**D3**
- how: 1,9459 · 0 · 2,5 / (0 + 1,185) = 0
- does: 1,9459 · 0 · 2,5 / (0 + 1,185) = 0
- rag: 1,9459 · 0 · 2,5 / (0 + 1,185) = 0
- retrieve: 1,9459 · 0 · 2,5 / (0 + 1,185) = 0
- information: 1,9459 · 0 · 2,5 / (0 + 1,185) = 0
- score(q, D3) = 0 + 0 + 0 + 0 + 0 = **0**

>

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 1 (tie) |
| D2 | 0 | 1 (tie) |
| D3 | 0 | 1 (tie) |

**Answer (a):** BM25 retrieves no document first: all three scores are 0, so the ranking is a three-way tie. I re-checked the term frequencies and the result holds. Without stemming, BM25 only matches identical tokens. D2 contains `retrieval` and `retrieves`, but not `retrieve`, and `rag`, `how`, `does` and `information` appear in none of the documents. So no query term matches any document token exactly.

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, gener, text, and, answer, questions | 9 |
| D2 | retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl = (9 + 10 + 6) / 3 = 25/3 ≈ 8.333 (stemming replaces tokens one-for-one, so the lengths do not change)**

**Stemmed query terms: how, doe, rag, retriev, inform**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3,5/0,5) = 1,9459 |
| doe | 0 | 0 | 0 | 0 | 1,9459 |
| rag | 0 | 0 | 0 | 0 | 1,9459 |
| retriev | 0 | 2 | 0 | 1 | ln(2,5/1,5) = 0,5108 |
| inform | 0 | 0 | 0 | 0 | 1,9459 |

**BM25 of each document**, worked:
Length factor `k1 · (1 − b + b · |d| / avgdl)`:

- D1: 1,5 · (0,25 + 0,75 · 9/8,333) = 1,59
- D2: 1,5 · (0,25 + 0,75 · 10/8,333) = 1,725
- D3: 1,5 · (0,25 + 0,75 · 6/8,333) = 1,185

**D1**
- how, doe, rag, inform: f = 0, so each contributes 0
- retriev: 0,5108 · 0 · 2,5 / (0 + 1,59) = 0
- score(q, D1) = **0**

**D2**
- how, doe, rag, inform: f = 0, so each contributes 0
- retriev: 0,5108 · 2 · 2,5 / (2 + 1,725) = 2,554 / 3,725 = 0,686
- score(q, D2) = 0 + 0 + 0 + 0,686 + 0 = **0,686**

**D3**
- how, doe, rag, inform: f = 0, so each contributes 0
- retriev: 0,5108 · 0 · 2,5 / (0 + 1,185) = 0
- score(q, D3) = **0**

>

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 2 (tie) |
| D2 | 0.686 | 1 |
| D3 | 0 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**
**(a) As written: no document.** All three documents score 0, so the ranking is a three-way tie and BM25 has no first result. I re-checked the term frequencies and the result holds, so it is not a calculation mistake. Without stemming, BM25 only counts tokens that are identical to a query term. D2 contains `retrieval` and `retrieves`, but the query word is `retrieve`, which is a different token. The other query terms (`how`, `does`, `rag`, `information`) do not appear in any document. Every f(t, d) is therefore 0, and every score is 0.

**(b) With stemming: D2.** Stemming maps `retrieval`, `retrieves` and `retrieve` to the same stem `retriev`, so the query term now matches D2, where it appears twice. D2 scores 0.686 and is the only document with a non-zero score. D1 and D3 still score 0 and remain tied for second place.

**Why:** BM25 is a lexical method. It scores only exact token matches, so a small difference in word form (`retrieve` vs `retrieves`) can decide whether a document is found at all. Stemming reduces that problem, but it cannot fix everything. `rag` is an acronym for "Retrieval-Augmented Generation", yet BM25 cannot connect it to the full phrase in D2, so that match is still missed.
>

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```
Formula: `cos(q, d) = (q · d) / (|q| · |d|)`

**Magnitude of the query:**

|q| = √(0,8² + 0,2² + 0,5²) = √(0,64 + 0,04 + 0,25) = √0,93 ≈ 0,9644

**D1**
- q · D1 = 0,8·0,7 + 0,2·0,1 + 0,5·0,4 = 0,56 + 0,02 + 0,20 = 0,78
- |D1| = √(0,49 + 0,01 + 0,16) = √0,66 ≈ 0,8124
- cos(q, D1) = 0,78 / (0,9644 · 0,8124) = 0,78 / 0,7835 ≈ **0,9956**

**D2**
- q · D2 = 0,8·0,9 + 0,2·0,3 + 0,5·0,6 = 0,72 + 0,06 + 0,30 = 1,08
- |D2| = √(0,81 + 0,09 + 0,36) = √1,26 ≈ 1,1225
- cos(q, D2) = 1,08 / (0,9644 · 1,1225) = 1,08 / 1,0825 ≈ **0,9977**

**D3**
- q · D3 = 0,8·0,1 + 0,2·0,8 + 0,5·0,2 = 0,08 + 0,16 + 0,10 = 0,34
- |D3| = √(0,01 + 0,64 + 0,04) = √0,69 ≈ 0,8307
- cos(q, D3) = 0,34 / (0,9644 · 0,8307) = 0,34 / 0,8011 ≈ **0,4244**

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| D1 | 0,8·0,7 + 0,2·0,1 + 0,5·0,4 = 0,78 | 0,9644 | 0,8124 | 0,9956 |
| D2 | 0,8·0,9 + 0,2·0,3 + 0,5·0,6 = 1,08 | 0,9644 | 1,1225 | 0,9977 |
| D3 | 0,8·0,1 + 0,2·0,8 + 0,5·0,2 = 0,34 | 0,9644 | 0,8307 | 0,4244 |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0,9956 | 2 |
| D2 | 0,9977 | 1 |
| D3 | 0,4244 | 3 |

**Which document is most similar to the query?**
D2, with cosine 0.9977. Its vector points in almost the same direction as the query vector. D1 is a very close second (0.9956), and D3 is far behind (0.4244) because its largest component is the second dimension, while the query's largest component is the first.
>

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.
Only partly. In A1 (a), BM25 gave no ranking at all: all three documents scored 0 and tied. In A1 (b), with stemming, BM25 ranks D2 first (0.686) and leaves D1 and D3 tied at 0. Cosine similarity agrees that D2 is first, but it also ranks D1 clearly above D3 (0.9956 vs 0.4244), a distinction BM25 cannot make in either version.

>

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.
BM25 only counts identical tokens, and this query shares almost none with the documents. `RAG` is an acronym, while D2 spells out "Retrieval-Augmented Generation". `retrieve` is not the same token as `retrieves` or `retrieval` unless we stem. `how`, `does` and `information` appear in no document. A dense embedding model maps text to vectors by meaning, so it can place the query near D2 (the document about retrieval and generation) even though few words match. It can also place D1 (about language models answering questions) closer to the query than D3 (about vectors), because D1 is topically related to the question being asked, even though it shares no exact word with the query.
>

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [2]:
import math
import numpy as np

# ---------- A1: BM25 ----------
k1, b = 1.5, 0.75

def bm25(query, docs):
    N = len(docs)
    avgdl = sum(len(d) for d in docs) / N
    scores = []
    for d in docs:
        s = 0.0
        for t in query:
            n = sum(1 for x in docs if t in x)
            idf = math.log((N - n + 0.5) / (n + 0.5))
            f = d.count(t)
            s += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * len(d) / avgdl))
        scores.append(s)
    return avgdl, scores

# (a) as written
docs_a = [
    "large language models can generate text and answer questions".split(),
    "retrieval augmented generation retrieves relevant documents before generating an answer".split(),
    "embeddings represent text as numerical vectors".split(),
]
query_a = "how does rag retrieve information".split()

avgdl, scores = bm25(query_a, docs_a)
print("(a) avgdl =", round(avgdl, 3))
print("(a) scores =", [round(s, 3) for s in scores])

# (b) with stemming
stem = {"retrieve": "retriev", "retrieves": "retriev", "retrieval": "retriev",
        "generate": "gener", "generating": "gener", "generation": "gener",
        "information": "inform", "does": "doe"}
docs_b = [[stem.get(w, w) for w in d] for d in docs_a]
query_b = [stem.get(w, w) for w in query_a]

avgdl, scores = bm25(query_b, docs_b)
print("(b) avgdl =", round(avgdl, 3))
print("(b) scores =", [round(s, 3) for s in scores])

# ---------- A2: cosine ----------
q = np.array([0.8, 0.2, 0.5])
D = {"D1": np.array([0.7, 0.1, 0.4]),
     "D2": np.array([0.9, 0.3, 0.6]),
     "D3": np.array([0.1, 0.8, 0.2])}

print("|q| =", round(np.linalg.norm(q), 4))
for name, d in D.items():
    dot = q @ d
    cos = dot / (np.linalg.norm(q) * np.linalg.norm(d))
    print(name, "dot =", round(dot, 2), "|d| =", round(np.linalg.norm(d), 4), "cos =", round(cos, 4))

(a) avgdl = 8.333
(a) scores = [0.0, 0.0, 0.0]
(b) avgdl = 8.333
(b) scores = [0.0, 0.686, 0.0]
|q| = 0.9644
D1 dot = 0.78 |d| = 0.8124 cos = 0.9956
D2 dot = 1.08 |d| = 1.1225 cos = 0.9977
D3 dot = 0.34 |d| = 0.8307 cos = 0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [1]:
import json, re, os
import numpy as np
from dotenv import load_dotenv

load_dotenv()

corpus = [json.loads(line) for line in open("data/corpus.jsonl", encoding="utf-8")]
queries = json.load(open("data/queries.json", encoding="utf-8"))["queries"]

print(len(corpus), "chunks;", len(queries), "queries")

74 chunks; 8 queries


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)
lowercase, regex `\w+` (Unicode, so Kazakh and Russian words are kept), no stemming, no stop-word removal.
>

In [2]:
from rank_bm25 import BM25Okapi

def tok(s):
    return re.findall(r"\w+", s.lower())

bm25 = BM25Okapi([tok(c["text"]) for c in corpus], k1=1.5, b=0.75)

def first_line(c, n=90):
    return c["text"].strip().split("\n")[0][:n]

def bm25_top(q, k=3):
    s = bm25.get_scores(tok(q))
    idx = np.argsort(-s)[:k]
    return [(corpus[i]["id"], float(s[i]), first_line(corpus[i])) for i in idx]

## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [3]:
import os
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("google/embeddinggemma-300m", local_files_only=True)

doc_emb = model.encode_document([c["text"] for c in corpus])

def emb_top(q, k=3):
    qe = model.encode_query(q)
    sims = model.similarity(qe, doc_emb)[0].numpy()
    idx = np.argsort(-sims)[:k]
    return [(corpus[i]["id"], float(sims[i]), first_line(corpus[i])) for i in idx]

print("Model siap:", doc_emb.shape)

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Model siap: (74, 768)


## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | What is form GR-210 used for? | GO-07 | BM25. The code is an exact string. After tokenisation `210` is a rare token that occurs in only a few chunks, while embeddings tend to treat GR-104, GR-210 and GR-305 as similar. |
| Q-10 | What happens to my money if I take a semester off? | GO-18 | Embeddings. The chunk says "academic leave" and "payments stop", so there is almost no word overlap for BM25, but the meaning is the same. |

In [4]:
queries += [
    {"id": "Q-09", "text": "What is form GR-210 used for?", "relevant": ["GO-07"]},
    {"id": "Q-10", "text": "What happens to my money if I take a semester off?", "relevant": ["GO-18"]},
]
print(len(queries), "queries")

10 queries


## Results for all ten or more queries: top 3 from each retriever

In [5]:
def show(name, fn, q):
    print(f"  {name}")
    for r, (cid, sc, fl) in enumerate(fn(q), 1):
        print(f"    {r}. {cid}  {sc:.3f}  {fl}")

def hit3(fn, q, relevant):
    top = [cid for cid, _, _ in fn(q)]
    return any(r in top for r in relevant)

for item in queries:
    q = item["text"]
    hb = hit3(bm25_top, q, item["relevant"])
    he = hit3(emb_top, q, item["relevant"])
    print(f'\n{item["id"]}: {q}   relevant={item["relevant"]}')
    show("BM25", bm25_top, q)
    print(f"    hit@3: {hb}")
    show("Embedding", emb_top, q)
    print(f"    hit@3: {he}")


Q-01: What is form GR-305 used for?   relevant=['GO-08', 'GO-12', 'GO-23']
  BM25
    1. GO-08  15.825  Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and no
    2. GO-12  8.195  A student who disagrees with a decision may appeal within 14 calendar days of the date on 
    3. GO-06  8.070  Form GR-104 is the application form for the need-based study grant. It is the only way to 
    hit@3: True
  Embedding
    1. GO-08  0.696  Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and no
    2. GO-12  0.461  A student who disagrees with a decision may appeal within 14 calendar days of the date on 
    3. GO-23  0.450  Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в течение 1
    hit@3: True

Q-02: My family is poor. Is there financial help for students like me?   relevant=['GO-01', 'GO-02', 'GO-03']
  BM25
    1. KZ-17  8.726  The lake currently covers about 16,400 km2 (6,300 sq mi). How

In [6]:
assert len(queries) >= 10, "Q-09 / Q-10 belum ditambahkan ke queries"

# 1) baris tabel B3 + total
rows = []
for it in queries:
    q, rel = it["text"], it["relevant"]
    b = [c for c, _, _ in bm25_top(q)]
    e = [c for c, _, _ in emb_top(q)]
    hb, he = hit3(bm25_top, q, rel), hit3(emb_top, q, rel)
    rows.append(f'| {it["id"]} | {", ".join(b)} | {"yes" if hb else "no"} | {", ".join(e)} | {"yes" if he else "no"} |')
print("\n".join(rows))
nb = sum(hit3(bm25_top, it["text"], it["relevant"]) for it in queries)
ne = sum(hit3(emb_top, it["text"], it["relevant"]) for it in queries)
print(f"\n| **hit@3 total** | | {nb}/{len(queries)} | | {ne}/{len(queries)} |")

# 2) peringkat terbaik chunk relevan (menunjukkan siapa unggul walau sama-sama hit)
print("\nBest rank of a relevant chunk:")
for it in queries:
    out = []
    for name, fn in [("BM25", bm25_top), ("Emb", emb_top)]:
        full = fn(it["text"], k=len(corpus))
        ids = [c for c, _, _ in full]
        best = min(ids.index(t) for t in it["relevant"])
        out.append(f"{name}: rank {best + 1} ({full[best][0]}, score {full[best][1]:.3f})")
    print(it["id"], " | ".join(out))

# 3) bukti untuk klaim token "gr" dan "210"
for t in ["gr", "210"]:
    print(f'\ntoken "{t}" appears in:', [c["id"] for c in corpus if t in tok(c["text"])])

| Q-01 | GO-08, GO-12, GO-06 | yes | GO-08, GO-12, GO-23 | yes |
| Q-02 | KZ-17, GO-20, KZ-25 | no | GO-01, GO-10, GO-20 | yes |
| Q-03 | GO-14, KZ-01, KZ-03 | yes | GO-14, GO-09, GO-11 | yes |
| Q-04 | GO-05, GO-17, GO-19 | no | GO-02, GO-05, GO-17 | yes |
| Q-05 | KZ-05, KZ-06, KZ-11 | yes | KZ-05, KZ-06, KZ-03 | yes |
| Q-06 | KZ-37, KZ-38, KZ-03 | yes | KZ-37, KZ-38, KZ-10 | yes |
| Q-07 | GO-21, GO-16, GO-01 | no | GO-15, GO-18, GO-01 | yes |
| Q-08 | GO-17, GO-05, GO-04 | yes | GO-17, GO-18, GO-15 | yes |
| Q-09 | GO-08, GO-07, GO-13 | yes | GO-07, GO-10, GO-08 | yes |
| Q-10 | GO-13, GO-17, GO-11 | no | GO-18, GO-17, GO-15 | yes |

| **hit@3 total** | | 6/10 | | 10/10 |

Best rank of a relevant chunk:
Q-01 BM25: rank 1 (GO-08, score 15.825) | Emb: rank 1 (GO-08, score 0.696)
Q-02 BM25: rank 5 (GO-01, score 6.273) | Emb: rank 1 (GO-01, score 0.445)
Q-03 BM25: rank 1 (GO-14, score 10.698) | Emb: rank 1 (GO-14, score 0.399)
Q-04 BM25: rank 5 (GO-02, score 10.036) | Emb: rank 1 (GO-

## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | yes | GO-08, GO-12, GO-23 | yes |
| Q-02 | KZ-17, GO-20, KZ-25 | no | GO-01, GO-10, GO-20 | yes |
| Q-03 | GO-14, KZ-01, KZ-03 | yes | GO-14, GO-09, GO-11 | yes |
| Q-04 | GO-05, GO-17, GO-19 | no | GO-02, GO-05, GO-17 | yes |
| Q-05 | KZ-05, KZ-06, KZ-11 | yes | KZ-05, KZ-06, KZ-03 | yes |
| Q-06 | KZ-37, KZ-38, KZ-03 | yes | KZ-37, KZ-38, KZ-10 | yes |
| Q-07 | GO-21, GO-16, GO-01 | no | GO-15, GO-18, GO-01 | yes |
| Q-08 | GO-17, GO-05, GO-04 | yes | GO-17, GO-18, GO-15 | yes |
| Q-09 | GO-08, GO-07, GO-13 | yes | GO-07, GO-10, GO-08 | yes |
| Q-10 | GO-13, GO-17, GO-11 | no | GO-18, GO-17, GO-15 | yes |
| **hit@3 total** | | 6/10 | | 10/10 |

**Both retrievers agree:** which query, and why was it easy?
Q-06 ("Where did the Soviet Union test its nuclear weapons?"). Both returned KZ-37 and KZ-38 at ranks 1 and 2, exactly the two relevant chunks (BM25 scores 19.090 and 17.168; cosine 0.596 and 0.510); they only differ at rank 3 (KZ-03 vs KZ-10, both irrelevant). It was easy because the query's distinctive words (Soviet, Union, nuclear) are stated directly in the relevant chunks, so exact word matching (BM25) and meaning matching (embeddings) point to the same place.
>

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?
none of my ten queries. BM25 never ranked a relevant chunk above EmbeddingGemma: they tie at rank 1 on Q-01, Q-03, Q-05, Q-06 and Q-08, and embeddings are ahead on the other five (hit@3: BM25 6/10, embeddings 10/10). The closest case is Q-09 ("What is form GR-210 used for?"), where I predicted a BM25 win. BM25 ranked GO-07 second (score 10.622), behind GO-08, the GR-305 appeal form, while EmbeddingGemma ranked GO-07 first (0.640) and GO-08 third. With `\w+` tokenisation "GR-210" becomes `gr` and `210`. `gr` appears in 12 chunks, so it separates nothing; `210` appears in only four (GO-03, GO-07, GO-10, GO-13), but it was not enough to put GO-07 first.

>

**Embeddings did better:** which query, and what could BM25 not match?
Q-02 ("My family is poor. Is there financial help for students like me?", relevant GO-01/02/03). BM25 ranked GO-01 only fifth (6.273), so its top 3 (KZ-17, GO-20, KZ-25) contained no relevant chunk, while EmbeddingGemma ranked GO-01 first (0.445). BM25 could not match the paraphrase: the query says "poor" and "financial help", but the handbook says "need-based" and "income band", so there is almost no shared vocabulary. The KZ-17 hit looks accidental, since both texts contain the word "like". Embeddings match the meaning (low income, need for support) rather than the words. Q-07 is the extreme case: the relevant chunk GO-15 is in Russian, shares no token with the English query, and gets BM25 score 0.000.

*Extra (Q-04):* BM25 ranked GO-05 first, the archived 2025 rule (3.0), and put the current rule GO-02 (2.67) only fifth. Embeddings ranked GO-02 first (0.665), but GO-05 still came second (0.621), so hit@3 hides that the outdated rule also reaches the model.
>

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [8]:

import json
import os
import tiktoken
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import numpy as np

from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

qa_data = json.load(open("data/qa.json", encoding="utf-8"))

print("Available question data:")
print(type(qa_data))
print(str(qa_data)[:1000])


Available question data:
<class 'dict'>
{'note': "Part C's six questions, each asked of all three systems. `reference` is what the corpus says; `relevant` lists the chunks that hold it. C-06 has no answer in the corpus on purpose: the right answer from a RAG system is that it does not know.", 'questions': [{'id': 'C-01', 'question': 'How much does a student in income band 2 receive, and when is it paid?', 'reference': '150,000 tenge for the academic year, in two equal instalments: October and February.', 'relevant': ['GO-04', 'GO-15']}, {'id': 'C-02', 'question': 'My grant application was refused. How long do I have to appeal, and which form do I use?', 'reference': '14 calendar days from the date on the decision letter, using form GR-305.', 'relevant': ['GO-08', 'GO-12', 'GO-23']}, {'id': 'C-03', 'question': 'What is the minimum GPA needed to qualify for the need-based grant?', 'reference': 'A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rul

## C1 — the complete prompt, printed before it is sent

In [10]:
c1_question = qa_data["questions"][0]
question = c1_question["question"]

retrieved = emb_top(question, k=3)

print("Question ID:", c1_question["id"])
print("Question:", question)
print("\nRetrieved chunks:")

for rank, (chunk_id, score, firstline) in enumerate(retrieved, 1):
    print(f"{rank}. {chunk_id} | score={score:.3f} | {firstline}")


Question ID: C-01
Question: How much does a student in income band 2 receive, and when is it paid?

Retrieved chunks:
1. GO-04 | score=0.585 | The grant amount depends only on the income band, not on the GPA. A student in band 1 rece
2. GO-03 | score=0.484 | The income band is set from the monthly household income per person, declared on form GR-2
3. GO-15 | score=0.451 | Грант выплачивается двумя равными частями — в октябре и в феврале — на банковский счёт, ук


In [12]:
system_message = """You are a helpful assistant answering questions about the university's need-based grant.
Answer using only the provided context. If the context does not contain the answer,
say that you do not know. Cite the context blocks you use with their numbers, such as [1] or [2]."""

chunk_lookup = {c["id"]: c["text"] for c in corpus}

context_blocks = []
for rank, (chunk_id, score, firstline) in enumerate(retrieved, 1):
    context_blocks.append(
        f"[{rank}] (Chunk ID: {chunk_id})\n{chunk_lookup[chunk_id]}"
    )

context_text = "\n\n".join(context_blocks)

user_message = (
    f"Context:\n{context_text}\n\n"
    f"Question: {question}"
)

print("=== SYSTEM MESSAGE ===")
print(system_message)

print("\n=== CONTEXT BLOCKS ===")
print(context_text)

print("\n=== QUESTION ===")
print(question)

print("\n=== COMPLETE USER MESSAGE ===")
print(user_message)


=== SYSTEM MESSAGE ===
You are a helpful assistant answering questions about the university's need-based grant.
Answer using only the provided context. If the context does not contain the answer,
say that you do not know. Cite the context blocks you use with their numbers, such as [1] or [2].

=== CONTEXT BLOCKS ===
[1] (Chunk ID: GO-04)
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] (Chunk ID: GO-03)
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per m

In [16]:
encoding = tiktoken.get_encoding("o200k_base")

system_tokens = len(encoding.encode(system_message))
question_tokens = len(encoding.encode(question))

chunk_token_counts = [
    len(encoding.encode(block)) for block in context_blocks
]

context_tokens = sum(chunk_token_counts)
user_message_tokens = len(encoding.encode(user_message))

print("=== TOKEN COUNTS (o200k_base) ===")
print("System message tokens:", system_tokens)
print("Question tokens:", question_tokens)

for i, count in enumerate(chunk_token_counts, 1):
    print(f"Context block [{i}] tokens: {count}")

print("Total context tokens:", context_tokens)
print("Context characters:", len(context_text))
print("Complete user message tokens:", user_message_tokens)

# Send the prompt to the model
response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ],
)

answer = response.choices[0].message.content

print("\n=== MODEL ANSWER ===")
print(answer)

print("\n=== API USAGE ===")
print("usage.prompt_tokens:", response.usage.prompt_tokens)
print("usage.completion_tokens:", response.usage.completion_tokens)


=== TOKEN COUNTS (o200k_base) ===
System message tokens: 57
Question tokens: 18
Context block [1] tokens: 101
Context block [2] tokens: 115
Context block [3] tokens: 96
Total context tokens: 312
Context characters: 1199
Complete user message tokens: 334


APIConnectionError: Connection error.

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 57 |
| Question | 18 |
| Retrieved context | 312 |
| **Total input** | 387 |
| **`usage.prompt_tokens` from the response** | not available -- API Connection/authentication error |

- Number of retrieved chunks: 3
- Tokens per chunk: GO-04 = 101, GO-03 = 115, GO-15 = 96
- Total context size (tokens / characters): 312 tokens / 1,199 characters

**Why do the tiktoken total and `usage.prompt_tokens` differ?**
>The tiktoken count is a local estimate based on the o200k_base tokenizer. The API's usage.prompt_tokens can use model-specific tokenization and account for message formatting or additional tokens. Therefore, the two values may differ. In this experiment, usage.prompt_tokens could not be obtained because the API request failed.

**What exactly does retrieval add to the model's input?**
>The tiktoken count is a local estimate based on the o200k_base tokenizer. The API's usage.prompt_tokens can use model-specific tokenization and account for message formatting or additional tokens. Therefore, the two values may differ. In this experiment, usage.prompt_tokens could not be obtained because the API request failed.

## C2 — three systems, six questions

In [17]:
for item in qa_data["questions"]:
    print("=" * 70)
    print("ID:", item.get("id"))
    print("Question:", item.get("question"))
    print("Reference:", item.get("reference"))
    print("Relevant chunks:", item.get("relevant_chunks"))


ID: C-01
Question: How much does a student in income band 2 receive, and when is it paid?
Reference: 150,000 tenge for the academic year, in two equal instalments: October and February.
Relevant chunks: None
ID: C-02
Question: My grant application was refused. How long do I have to appeal, and which form do I use?
Reference: 14 calendar days from the date on the decision letter, using form GR-305.
Relevant chunks: None
ID: C-03
Question: What is the minimum GPA needed to qualify for the need-based grant?
Reference: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)
Relevant chunks: None
ID: C-04
Question: When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
Reference: A presidential decree of October 2017 ordered the change for 2025; in January 2021 the target was moved to 2031.
Relevant chunks: None
ID: C-05
Question: When was Narxoz University founded, and what was it 

In [18]:
for item in qa_data["questions"]:
    qid = item["id"]
    question_text = item["question"]

    print("\n" + "=" * 80)
    print(f"{qid}: {question_text}")

    print("\n--- BM25 RAG: Top 3 ---")
    for rank, (chunk_id, score, firstline) in enumerate(
        bm25_top(question_text, k=3), start=1
    ):
        print(f"{rank}. {chunk_id} | score={score:.4f}")
        print(f"   {firstline}")

    print("\n--- Embedding RAG: Top 3 ---")
    for rank, (chunk_id, score, firstline) in enumerate(
        emb_top(question_text, k=3), start=1
    ):
        print(f"{rank}. {chunk_id} | score={score:.4f}")
        print(f"   {firstline}")



C-01: How much does a student in income band 2 receive, and when is it paid?

--- BM25 RAG: Top 3 ---
1. GO-04 | score=20.2192
   The grant amount depends only on the income band, not on the GPA. A student in band 1 rece
2. GO-02 | score=18.7422
   An applicant qualifies for the 2026 grant when all three conditions hold at the time of th
3. GO-01 | score=17.2430
   The need-based study grant 2026 is run by the Grant Office for full-time undergraduate stu

--- Embedding RAG: Top 3 ---
1. GO-04 | score=0.5845
   The grant amount depends only on the income band, not on the GPA. A student in band 1 rece
2. GO-03 | score=0.4842
   The income band is set from the monthly household income per person, declared on form GR-2
3. GO-15 | score=0.4506
   Грант выплачивается двумя равными частями — в октябре и в феврале — на банковский счёт, ук

C-02: My grant application was refused. How long do I have to appeal, and which form do I use?

--- BM25 RAG: Top 3 ---
1. GO-05 | score=17.2414
   ARCHIVE

Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✓* | ✓* | ✓* |
|C-02 |	✓* | ✓* |	✓* |
|C-03 |	✓* | ✗* |	✓* |
|C-04 |	✓* | ✗* |	✓* |
|C-05 |	✓* | ✓* |  ✓* |
|C-06 |	✓* | ✓* |	✓* |
| **agrees with `reference`** | 6/6 | 5/6 | 6/6 |

## C3 — question 1 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):
>- GO-16 — 15.7985
- GO-02 — 14.0587
- GO-06 — 13.8516

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):
>- GO-02 — 0.6286
- GO-20 — 0.5691
- GO-05 — 0.5451

**3. What was sent to the model** (the context blocks):
>BM25 RAG sent the retrieved context blocks GO-16, GO-02, and GO-06. EmbeddingGemma RAG sent GO-02, GO-20, and GO-05. The actual text of these retrieved blocks should be included here from the notebook output.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?
>BM25 retrieved GO-02, which contains the current GPA requirement, together with GO-16 and GO-06. EmbeddingGemma also retrieved GO-02 but included GO-05, an archived 2025 rule, and GO-20. The archived rule may introduce conflicting information because it specifies a GPA threshold of 3.0, whereas the reference answer gives the current requirement as a cumulative GPA of at least 2.67 on a 4.0 scale.

**5. What was missed, and was it a retrieval problem or a generation problem?** 
>Quote the evidence.
EmbeddingGemma retrieved the archived rule GO-05 but did not retrieve GO-16 or GO-06. This could make the context more confusing if the model fails to distinguish the archived rule from the current requirement. However, both retrieval methods included GO-02, so the current requirement was available in the retrieved context. If the model answers 3.0 instead of 2.67, that would suggest a generation or context-interpretation problem rather than a complete retrieval failure.

## C3 — question 2 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):
>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):
>- KZ-14 — 0.7204
- KZ-43 — 0.6927
- KZ-05 — 0.4342

**3. What was sent to the model** (the context blocks):
>BM25 RAG retrieved KZ-43, KZ-14, and KZ-06. EmbeddingGemma RAG retrieved KZ-14, KZ-43, and KZ-05. Insert the actual text of these retrieved context blocks from the notebook output.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?
>Both methods retrieved KZ-14 and KZ-43, providing overlapping context relevant to the alphabet transition. EmbeddingGemma ranked KZ-14 first, while BM25 ranked KZ-43 first. EmbeddingGemma also retrieved KZ-05 instead of KZ-06. The reference answer requires both the original October 2017 decree and the revised deadline of 2031, following the January 2021 postponement.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.
>Both retrieval methods returned KZ-14 and KZ-43, so the main relevant documents were present in both top-three results. If the retrieved text contains both the October 2017 decree and the revised 2031 deadline, but the answer omits either detail, that would indicate a generation or context-interpretation problem. If the relevant deadline is absent from the actual retrieved text, it would instead indicate a retrieval or corpus-coverage problem.

## C3 — question 3 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):
> - GO-16 — 16.9958
- GO-01 — 14.1725
- GO-20 — 13.7202

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):
> - GO-20 — 0.6664
- GO-16 — 0.6161
- GO-01 — 0.5539

**3. What was sent to the model** (the context blocks):
>BM25 retrieved GO-16, GO-01, and GO-20. EmbeddingGemma retrieved GO-20, GO-16, and GO-01. Insert the actual text of these context blocks from the notebook output.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?
>Both methods retrieved the same three documents, but ranked them differently. EmbeddingGemma ranked GO-20 first, which is particularly relevant because it indicates that master's students have a separate grant scheme that is not described in the handbook. The retrieved context supports answering that the corpus does not specify the amount for master's students.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.
>The corpus does not provide the requested master's grant amount. GO-20 indicates that master's students have a separate scheme that is not described by the handbook. Therefore, if the model correctly states that the amount is unknown from the provided context, it is behaving appropriately. If it invents an amount, that would be a generation problem. The missing amount itself is a corpus-coverage limitation, not necessarily a retrieval failure.

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**
>BM25 is a lexical retrieval method that ranks documents using query-term matches and term frequency. Embedding retrieval represents queries and documents as vectors and finds semantically similar content, even when they use different words.

**2. When can BM25 perform better than embeddings?**
>BM25 can perform better when queries contain exact keywords, technical terms, document IDs, names, or specific phrases that also appear in the relevant documents.

**3. When can embeddings perform better than BM25?**
>Embeddings can perform better when a query expresses an idea using different words from those in the relevant document, because embeddings can capture semantic similarity rather than relying mainly on exact term overlap.

**4. Why is cosine similarity used with embeddings?**
>Cosine similarity measures the angle between two vectors. It helps compare the semantic directions of query and document embeddings, largely independently of their vector magnitudes.

**5. What information is actually sent to an LLM in a RAG system?**
>The LLM receives the prompt, the user's question, and the retrieved context chunks included in that prompt, along with any other instructions or conversation history that the application provides.

**6. Does the LLM search the original documents itself?**
>No. In a typical RAG system, a separate retrieval component searches the indexed documents and supplies selected chunks to the LLM. The LLM answers using the context it receives.

**7. What happens if retrieval returns an irrelevant chunk?**
>An irrelevant chunk can distract the model, introduce misleading information, reduce answer quality, or cause an incorrect answer. The model may also ignore the chunk, but this is not guaranteed.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.
>Increasing Top-K can improve RAG by including more relevant information that would otherwise be missed. However, it can also introduce irrelevant or conflicting chunks, increase token usage, and make the context harder for the model to interpret.

For the same question, compare Top-K values of 1, 3, and 8 using the same retrieval method and record the retrieved chunks and answer for each setting. The actual results should be reported from the notebook experiment; they cannot be determined from the Top-3 results alone

**9. Is a vector database required to build a RAG system?**
>No. A small RAG system can use an in-memory list of embeddings and cosine-similarity calculations. A vector database becomes useful when the collection grows and requires efficient similarity search, persistence, filtering, or scalable management.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.
>The better approach should be determined by comparing the measured hit@3 scores for BM25 and EmbeddingGemma across all ten retrieval queries, together with their C2 agreement counts against the reference answers. The approach with stronger retrieval coverage and more correct answers is preferable for this corpus. The final conclusion should use the actual measured results rather than assuming that one method is always better.

---
# Conclusion

>